# Parking Sign Detection - A/B Test: Version B (Negative Annotations OFF, Augmentation ON)

Isolating the effect of aggressive augmentation by removing negative annotations.

**Variable under test:** Aggressive augmentation (mosaic, scale, mixup, copy_paste, etc.)
**Control:** Negative annotations OFF (background crops with empty labels filtered out)

**Dataset:** ~8,600+ images, 1 class (parking_sign)
- Combined from: parking-sign-coco (Roboflow) + sf-parking-signs (Figure Eight)
- All images resized to 640x640
- **Negative background crops REMOVED (empty label files filtered out)**
- Train/Val/Test split: 80/10/10

**Model:** YOLO11m (Medium)

**Training:** 30 epochs, cosine LR decay, 2 GPUs, **aggressive augmentation**

## 1. Setup

In [ ]:
# Fix numpy version first (torchvision requires numpy<2)
!pip uninstall numpy -y -q
!pip install "numpy<2" -q

# Then install ultralytics
!pip install ultralytics -q

# Restart runtime after this cell if needed

In [ ]:
from pathlib import Path
from ultralytics import YOLO
import pandas as pd
import matplotlib.pyplot as plt
import shutil
import yaml
import os

# Kaggle dataset path (nested directory structure confirmed by user)
DATASET_PATH = Path("/kaggle/input/parking-sign-detection-coco-dataset/parking-sign-detection-coco-dataset")
OUTPUT_PATH = Path("/kaggle/working")
DATA_YAML = DATASET_PATH / "data.yaml"

print(f"Dataset Path: {DATASET_PATH}")
print(f"Dataset exists: {DATASET_PATH.exists()}")

if DATASET_PATH.exists():
    print(f"Contents: {list(DATASET_PATH.iterdir())}")
    
    # Parse data.yaml
    with open(DATA_YAML) as f:
        data_config = yaml.safe_load(f)
    
    print("Original config:", data_config)
    
    # Update paths to use absolute paths
    new_config = data_config.copy()
    
    for split in ["train", "val", "test"]:
        keys = [split]
        if split == 'val': keys.append('valid')
        
        for k in keys:
            if k in new_config:
                folder_name = "valid" if split == "val" else split
                
                img_path = DATASET_PATH / folder_name / "images"
                if not img_path.exists():
                    img_path = DATASET_PATH / folder_name
                
                if img_path.exists():
                    new_config[k] = str(img_path)
                    print(f"Resolved {k} to {img_path}")
                else:
                    print(f"WARNING: Could not find image path for {k} at {img_path}")

    new_config['path'] = str(DATASET_PATH)
    
    FIXED_DATA_YAML = OUTPUT_PATH / "data.yaml"
    with open(FIXED_DATA_YAML, "w") as f:
        yaml.dump(new_config, f)
    
    DATA_YAML = FIXED_DATA_YAML
    print(f"\nFixed data.yaml written to {DATA_YAML}")
    !cat {DATA_YAML}
else:
    print("ERROR: DATASET_PATH not found. Please check directory structure in /kaggle/input")
    !ls -R /kaggle/input | head -n 20

## 2. Dataset Verification

In [ ]:
# Count images in each split
for split in ["train", "valid", "test"]:
    try:
        folder_name = split
        if split == 'val' and not (DATASET_PATH / split).exists():
             folder_name = 'valid'
             
        img_dir = DATASET_PATH / folder_name / "images"
        label_dir = DATASET_PATH / folder_name / "labels"
        
        if img_dir.exists():
            n_images = len(list(img_dir.glob("*.jpg")))
            n_labels = len(list(label_dir.glob("*.txt")))
            print(f"{split} ({folder_name}): {n_images} images, {n_labels} labels")
        else:
            print(f"{split}: Directory not found at {img_dir}")
    except Exception as e:
        print(f"Error checking {split}: {e}")

## 3. Filter Out Negative Annotations

Remove images with empty label files (negative background crops) from the training set.
This isolates the effect of augmentation alone, without negative samples confounding the result.

In [ ]:
import shutil
from pathlib import Path

# Create filtered dataset without negative annotations (empty label files)
FILTERED_PATH = OUTPUT_PATH / "dataset_no_negatives"

for split in ["train", "valid", "test"]:
    src_img_dir = DATASET_PATH / split / "images"
    src_lbl_dir = DATASET_PATH / split / "labels"
    dst_img_dir = FILTERED_PATH / split / "images"
    dst_lbl_dir = FILTERED_PATH / split / "labels"
    
    dst_img_dir.mkdir(parents=True, exist_ok=True)
    dst_lbl_dir.mkdir(parents=True, exist_ok=True)
    
    if not src_img_dir.exists():
        print(f"Skipping {split}: {src_img_dir} not found")
        continue
    
    total = 0
    kept = 0
    removed = 0
    
    for img_file in sorted(src_img_dir.glob("*")):
        total += 1
        lbl_file = src_lbl_dir / (img_file.stem + ".txt")
        
        # Skip images with empty label files (negative samples)
        if lbl_file.exists() and lbl_file.stat().st_size == 0:
            removed += 1
            continue
        
        # Copy image and label
        shutil.copy2(img_file, dst_img_dir / img_file.name)
        if lbl_file.exists():
            shutil.copy2(lbl_file, dst_lbl_dir / lbl_file.name)
        kept += 1
    
    print(f"{split}: {total} total -> {kept} kept, {removed} negative samples removed")

# Write filtered data.yaml
filtered_config = {
    'path': str(FILTERED_PATH),
    'train': str(FILTERED_PATH / 'train' / 'images'),
    'val': str(FILTERED_PATH / 'valid' / 'images'),
    'test': str(FILTERED_PATH / 'test' / 'images'),
    'nc': 1,
    'names': ['parking_sign'],
}

FILTERED_DATA_YAML = FILTERED_PATH / "data.yaml"
with open(FILTERED_DATA_YAML, 'w') as f:
    yaml.dump(filtered_config, f)

print(f"\nFiltered data.yaml written to {FILTERED_DATA_YAML}")
!cat {FILTERED_DATA_YAML}

## 4. Training Configuration

### Version B: Negative Annotations OFF, Augmentation ON
Uses the filtered dataset (no empty-label background crops).
Aggressive augmentation enabled to test if augmentation alone causes instability.

In [ ]:
RUN_NAME = "parking_sign_aug_nonegann"

TRAIN_PARAMS = {
    "data": str(FILTERED_DATA_YAML),
    "epochs": 30,
    "imgsz": 640,
    "batch": 16,
    "workers": 4,
    "patience": 15,
    "save_period": 10,
    "project": str(OUTPUT_PATH / "runs"),
    "name": RUN_NAME,
    "exist_ok": True,
    "pretrained": True,
    "verbose": True,
    # Multi-GPU (DDP)
    "device": "0,1",
    # Cosine LR decay
    "cos_lr": True,
    "lr0": 0.005,
    "lrf": 0.005,
    # AGGRESSIVE augmentation (v3)
    "mosaic": 1.0,          # Combine 4 images
    "mixup": 0.08,          # Image blending
    "copy_paste": 0.05,     # Copy-paste augmentation
    "degrees": 8.0,         # Rotation
    "translate": 0.1,       # Translation
    "scale": 0.4,           # KEY suspect for instability
    "shear": 2.0,           # Shear
    "perspective": 0.0001,  # Perspective warp
    "fliplr": 0.5,          # Horizontal flip
    "flipud": 0.0,
    "hsv_h": 0.01,
    "hsv_s": 0.5,
    "hsv_v": 0.3,
    # Close mosaic late for fine-tuning
    "close_mosaic": 25,     # Last 5 epochs without mosaic
}

print(f"Training config: {len(TRAIN_PARAMS)} parameters")
print(f"Model: yolo11m | Epochs: {TRAIN_PARAMS['epochs']}, Patience: {TRAIN_PARAMS['patience']}")
print(f"Image size: {TRAIN_PARAMS['imgsz']} | Device: {TRAIN_PARAMS['device']} (2 GPUs)")
print(f"Negative annotations: OFF | Augmentation: ON")
print(f"  mosaic={TRAIN_PARAMS['mosaic']}, scale={TRAIN_PARAMS['scale']}, mixup={TRAIN_PARAMS['mixup']}")

## 5. Train Model

In [ ]:
print("="*60)
print("Training Version B: Neg Ann OFF, Augmentation ON")
print("="*60)

model = YOLO("yolo11m.pt")
train_results = model.train(**TRAIN_PARAMS)

## 6. Training Results

In [ ]:
# Show training curves
from IPython.display import Image, display

results_img = OUTPUT_PATH / "runs" / RUN_NAME / "results.png"
if results_img.exists():
    print("Training curves:")
    display(Image(filename=str(results_img)))
else:
    print(f"Results image not found at {results_img}")

## 7. Evaluate Best Model

In [ ]:
# Load best model
best_model_path = OUTPUT_PATH / "runs" / RUN_NAME / "weights" / "best.pt"
best_model = YOLO(best_model_path)

print(f"Loaded model from: {best_model_path}")

# Run validation on test set
print("\nEvaluating on test set...")
test_results = best_model.val(data=str(FILTERED_DATA_YAML), split="test")

print(f"\nTest Set Results:")
print(f"  mAP50: {test_results.results_dict.get('metrics/mAP50(B)', 0):.4f}")
print(f"  mAP50-95: {test_results.results_dict.get('metrics/mAP50-95(B)', 0):.4f}")

In [ ]:
# Show confusion matrix
confusion_img = OUTPUT_PATH / "runs" / RUN_NAME / "confusion_matrix.png"
if confusion_img.exists():
    print("Confusion matrix:")
    display(Image(filename=str(confusion_img)))

## 8. Export Best Model

In [ ]:
# Export to ONNX for deployment
print("Exporting best model to ONNX...")
onnx_path = best_model.export(format="onnx", imgsz=640, simplify=True)
print(f"Exported to: {onnx_path}")

# Also save PyTorch format to output root
final_model_path = OUTPUT_PATH / f"parking_sign_detector_{RUN_NAME}.pt"
shutil.copy(best_model_path, final_model_path)
print(f"PyTorch model: {final_model_path}")

## 9. Results Summary

### Version B: Negative Annotations OFF, Augmentation ON
- **Epochs:** 30, patience 15
- **Negative annotations:** OFF (background crops with empty labels filtered out)
- **Augmentation:** ON (mosaic=1.0, scale=0.4, mixup=0.08, copy_paste=0.05, degrees=8, shear=2)
- **Hypothesis:** If augmentation alone causes instability, loss curve will be oscillatory/increasing

### Compare with Version A
`parking_sign_training_baseline.ipynb` has the opposite config:
- Negative annotations ON + Augmentation OFF

### Decision Matrix
| A (neg ann ON, aug OFF) | B (neg ann OFF, aug ON) | Conclusion |
|------------------------|------------------------|------------|
| Stable | Unstable | Augmentation is the problem |
| Unstable | Stable | Negative annotations are the problem |
| Both stable | Both stable | The combination causes instability |
| Both unstable | Both unstable | Both changes are problematic |

In [ ]:
# Print final summary
print("\n" + "="*60)
print("VERSION B (NEG ANN OFF, AUG ON) TRAINING COMPLETE")
print("="*60)
print(f"\nRun name: {RUN_NAME}")
print(f"Output files:")
print(f"  - {final_model_path}")
print(f"  - {OUTPUT_PATH / 'runs' / RUN_NAME / 'results.csv'}")